# GPT-5 — Data Collection (Reproducibility)

This notebook collects the **reproducibility** of the GPT-5 response corpus used in the paper — 5 Chain-of-Thought variants per base question, 30 runs per variant.

**Protocol (identical to the Gemini 3 Flash collection notebook):**
- Model: `gpt-5` (OpenAI; released August 7, 2025)
- Temperature: 1.0, top_p: 1.0
- Runs per (prompt, variant): 30
- CoT variants per prompt: 5 (v1 Traditional, v2 Differential Diagnosis, v3 Intuitive, v4 Analytic, v5 Bayesian — see Methods §2.2 of the paper)
- Max concurrent requests: 5 (asyncio semaphore)
- Per-(prompt, variant) inter-run spacing: 60 seconds within the same (prompt, variant)
- Up to 6 retry attempts per API call
- Continuation logic if the response ends with `finish_reason == 'length'` (up to 2 extra calls with "Continue.")
- Parallel across the 75 (prompt × variant) pairs; sequential across the 30 runs within a pair


> **Reviewer note:** To reproduce this notebook, update `PROJECT_DIR` to your local path. API keys are read from a `.env` file in `PROJECT_DIR` (see the `.env.example` in the repository root).

In [ ]:
# ---------- Load API key from .env (GitHub release version) ----------
# The hardcoded API key has been replaced with an environment-variable read.
# Place a .env file in PROJECT_DIR containing:  OPENAI_API_KEY=sk-...
import os
from pathlib import Path

PROJECT_DIR = Path('/Users/luizzadallapria/Documents/GPT5_Study')

def load_env(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            if '=' in line:
                k, v = line.split('=', 1)
                k = k.strip()
                if k.lower() == 'openai_api_key':
                    k = 'OPENAI_API_KEY'
                os.environ[k] = v.strip().strip('"').strip("'")

load_env(PROJECT_DIR / '.env')
assert os.environ.get('OPENAI_API_KEY'), 'OPENAI_API_KEY missing from .env'
print('API key loaded ✓')


In [ ]:
from openai import OpenAI
import pandas as pd
import time
import os
import csv
import datetime

# ---------------
# Amelia Changes
# ---------------
import threading
import concurrent.futures
import asyncio
# this next line can be added with the earlier one once we decide on the version using
from openai import AsyncOpenAI
# ---------------

In [ ]:
# Luizza added downloading csv with prompts
df_prompts = pd.read_csv("/content/new_reproductibility_prompts.csv")

# Same structure as before:
# { prompt_id: [ (variant_id, variant_text), ... ], ... }
PROMPTS_VARIANTS = (
    df_prompts
    .groupby("prompt_id")
    .apply(lambda g: list(zip(g["variant_id"], g["variant_text"])))
    .to_dict()
)

/tmp/ipython-input-1405399244.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: list(zip(g["variant_id"], g["variant_text"])))


In [ ]:
# CONFIG
MODEL_NAME   = "gpt-5"      # GPT model
TEMPERATURE = 1
NUM_RUNS     = 30           # per variant
CSV_PATH     = "reproducibility_results.csv"

# -----------------
# amelia's changes
# -----------------
# max concurrent requests that can hit the Chat API at once
MAX_CONCURRENT_REQUESTS = 5

# Initialize client
# ---------------
# Amelia's changes
# -----------
client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])


# CSV header
header = [
    "prompt_id","variant_id","variant_text","run_id",
    "model",
    "timestamp_utc","latency_ms",
    "prompt_tokens","completion_tokens","total_tokens",
    "finish_reason","answer_content"
]

# Make sure CSV exists with header
if not os.path.exists(CSV_PATH):
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(header)

# ----------------
# Amelia's changes
# ----------------
async def call_model(prompt_text, sem: asyncio.Semaphore):
    """
    Async version of call_model, main changes are:
    - continuation if finish_reason == 'length'
    - per-request concurrency control via semaphore
    """
    async with sem: ### Luizza added this line ###
        t0 = time.time()
        resp = await client.chat.completions.create(
            model=MODEL_NAME,
            temperature=TEMPERATURE,
            top_p=1.0,
            messages=[
                {"role": "user", "content": prompt_text}
            ]
        )
        t1 = time.time()



    choice = resp.choices[0]
    usage  = getattr(resp, "usage", None)

    return {
        "answer": choice.message.content,
        "finish_reason": getattr(choice, "finish_reason", None),
        "latency_ms": int((t1 - t0) * 1000),
        "prompt_tokens": getattr(usage, "prompt_tokens", None),
        "completion_tokens": getattr(usage, "completion_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
    }

async def run_varient(
    prompt_id: int,
    variant_id: str,
    variant_text: str,
    sem: asyncio.Semaphore,
    csv_lock: asyncio.Lock,
):
    """
    Runs NUM_RUNS for a single (prompt_id, prompt_text),
    ensuring at least 5 seconds between runs of THIS prompt.

    This is the part that is working directly on managing the inputs and outputs of the parallelization
    """
    print(f"\n          prompt {prompt_id}:{variant_id} collecting {NUM_RUNS} runs")


    # cycling through the runs, and keeping track of everything
    for run_id in range(1, NUM_RUNS + 1):
        try:
            # running the main for loop to get the prompt
            out = await call_model(variant_text, sem)

            # if utcnow is giving you problems, just switch it to time(), which I think should still work
            row = [
                prompt_id, variant_id, variant_text, run_id,
                MODEL_NAME,
                datetime.datetime.utcnow().isoformat(),
                out["latency_ms"],
                out["prompt_tokens"], out["completion_tokens"], out["total_tokens"],
                out["finish_reason"], out["answer"],
            ]

            # single-writer section for CSV (Important to keep here when locking is happening so no overwrites)
            async with csv_lock:
                with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
                    csv.writer(f).writerow(row)

            # printing if the run was successful
            print(
                f"Successful prompt {prompt_id}:{variant_id} Run {run_id:02d} saved (latency {out['latency_ms']} ms, tokens {out['total_tokens']})"
            )

        # trying again if fails
        except Exception as e:
            print(f"Failed Prompt and will repeat soon; {prompt_id}:{variant_id} Run {run_id:02d} error: {e}")
            # light backoff on error!!!! Change this to 60 if keep running into errors
            await asyncio.sleep(5)
            continue

        # enforceing >= 60 seconds between varients... you can remove this, but I am not sure if you want to
        # this will also help with making sure not to overload the amount of calls you can make
        await asyncio.sleep(60)


async def run_entire_code():
    """
    this is basically the main function

    we just broke all the parts that are called commonly down so its easier to read, feel free to revert this if you want

    this is also the mechanism that controls how many concurrent requests are, and when the locking and gathering takes place
    """
    sem = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
    csv_lock = asyncio.Lock()

    tasks = []
    # this part is your main loop!!!
    # the rest of the for loop (the real bones) of the for loop is in the run_varient function
    for prompt_id, variants in PROMPTS_VARIANTS.items():
        for variant_id, variant_text in variants:
            tasks.append(
                asyncio.create_task(
                    run_varient(prompt_id, variant_id, variant_text, sem, csv_lock)
                )
            )

    await asyncio.gather(*tasks)
    print(f"\n✅ Done. Results saved to {CSV_PATH}")

# running everything
await run_entire_code()



          prompt 1: collecting 30 runs

          prompt 1: collecting 30 runs

          prompt 1: collecting 30 runs

          prompt 1: collecting 30 runs

          prompt 1: collecting 30 runs

          prompt 2: collecting 30 runs

          prompt 2: collecting 30 runs

          prompt 2: collecting 30 runs

          prompt 2: collecting 30 runs

          prompt 2: collecting 30 runs

          prompt 3: collecting 30 runs

          prompt 3: collecting 30 runs

          prompt 3: collecting 30 runs

          prompt 3: collecting 30 runs

          prompt 3: collecting 30 runs

          prompt 4: collecting 30 runs

          prompt 4: collecting 30 runs

          prompt 4: collecting 30 runs

          prompt 4: collecting 30 runs

          prompt 4: collecting 30 runs

          prompt 5: collecting 30 runs

          prompt 5: collecting 30 runs

          prompt 5: collecting 30 runs

          prompt 5: collecting 30 runs

          prompt 5: collecting 30 runs


C:\Users\ameli\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\httpcore\_async\connection_pool.py:284: RuntimeWarning: coroutine 'AsyncCompletions.create' was never awaited
  for connection in list(self._connections):


Successful prompt 11:v3 Run 01 saved (latency 31632 ms, tokens 1623)
Successful prompt 9:v1 Run 01 saved (latency 35503 ms, tokens 1875)
Successful prompt 12:v3 Run 01 saved (latency 35768 ms, tokens 1767)
Successful prompt 7:v3 Run 01 saved (latency 37963 ms, tokens 1926)
Successful prompt 9:v3 Run 01 saved (latency 39389 ms, tokens 1970)
Successful prompt 12:v4 Run 01 saved (latency 40590 ms, tokens 2099)
Successful prompt 12:v2 Run 01 saved (latency 41528 ms, tokens 2060)
Successful prompt 4:v2 Run 01 saved (latency 44111 ms, tokens 2436)
Successful prompt 11:v1 Run 01 saved (latency 44173 ms, tokens 2268)
Successful prompt 5:v1 Run 01 saved (latency 44524 ms, tokens 2416)
Successful prompt 15:v1 Run 01 saved (latency 44727 ms, tokens 2149)
Successful prompt 4:v3 Run 01 saved (latency 45107 ms, tokens 2486)
Successful prompt 1:v1 Run 01 saved (latency 45192 ms, tokens 2716)
Successful prompt 11:v5 Run 01 saved (latency 46390 ms, tokens 2699)
Successful prompt 6:v3 Run 01 saved (late

CancelledError: 

In [ ]:
# -------- CONFIG --------
MODEL_NAME = "gpt-5"
TEMPERATURE = 1
NUM_RUNS = 30
CSV_PATH = "reproducibility_results.csv"
MAX_CONCURRENT_REQUESTS = 5

# Use env var for key (set it in your environment)
client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])

# CSV header
header = [
    "prompt_id","variant_id","variant_text","run_id",
    "model","temperature",
    "timestamp_utc","latency_ms",
    "prompt_tokens","completion_tokens","total_tokens",
    "finish_reason","answer_content"
]

if not os.path.exists(CSV_PATH):
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        csv.writer(f, quoting=csv.QUOTE_ALL).writerow(header)

def _extract_text(choice):
    txt = getattr(choice.message, "content", None)
    if isinstance(txt, str) and txt.strip():
        return txt.strip()
    if isinstance(txt, list):
        parts = []
        for part in txt:
            if isinstance(part, dict):
                if isinstance(part.get("text"), str):
                    parts.append(part["text"])
                elif isinstance(part.get("content"), str):
                    parts.append(part["content"])
        return "\n".join(parts).strip() if parts else ""
    return ""

async def call_model(prompt_text: str, sem: asyncio.Semaphore):
    """
    One logical call. Optional continuation if output is cut off.
    """
    t0 = time.time()

    messages = [{"role": "user", "content": prompt_text}]
    collected = []

    total_prompt_tokens = 0
    total_completion_tokens = 0
    total_tokens = 0

    finish_reason = None
    MAX_CONTINUES = 2
    REQUEST_TIMEOUT = 200

    async with sem:
        for _ in range(MAX_CONTINUES + 1):
            resp = await client.chat.completions.create(
                model=MODEL_NAME,
                temperature=TEMPERATURE,
                top_p=1.0,
                messages=messages,
                timeout=REQUEST_TIMEOUT,
                # If your SDK supports it, set a high max output to reduce truncation:
            )

            choice = resp.choices[0]
            usage = getattr(resp, "usage", None)

            text = _extract_text(choice)
            collected.append(text)

            if usage:
                total_prompt_tokens     += getattr(usage, "prompt_tokens", 0) or 0
                total_completion_tokens += getattr(usage, "completion_tokens", 0) or 0
                total_tokens            += getattr(usage, "total_tokens", 0) or 0

            finish_reason = getattr(choice, "finish_reason", None)
            if finish_reason != "length":
                break

            messages.append({"role": "assistant", "content": text})
            messages.append({"role": "user", "content": "Continue."})

    t1 = time.time()
    final_text = "\n".join([c for c in collected if c]).strip()

    return {
        "answer": final_text,
        "finish_reason": finish_reason,
        "latency_ms": int((t1 - t0) * 1000),
        "prompt_tokens": total_prompt_tokens,
        "completion_tokens": total_completion_tokens,
        "total_tokens": total_tokens,
    }

async def run_variant(prompt_id, variant_id, variant_text, sem, csv_lock):
    print(f"\nprompt {prompt_id}:{variant_id} collecting {NUM_RUNS} runs")

    for run_id in range(1, NUM_RUNS + 1):
        retries = 0
        MAX_RETRIES = 6

        while True:
            try:
                out = await call_model(variant_text, sem)

                row = [
                    prompt_id, variant_id, variant_text, run_id,
                    MODEL_NAME, TEMPERATURE,
                    datetime.datetime.utcnow().isoformat(),
                    out["latency_ms"],
                    out["prompt_tokens"], out["completion_tokens"], out["total_tokens"],
                    out["finish_reason"], out["answer"],
                ]

                async with csv_lock:
                    with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
                        csv.writer(f, quoting=csv.QUOTE_ALL).writerow(row)

                print(f"Saved {prompt_id}:{variant_id} run {run_id:02d} (tokens {out['total_tokens']})")
                break

            except Exception as e:
                retries += 1
                print(f"Error {prompt_id}:{variant_id} run {run_id:02d} attempt {retries}: {e}")

                if retries >= MAX_RETRIES:
                    print(f"Giving up on {prompt_id}:{variant_id} run {run_id:02d} after {MAX_RETRIES} attempts")
                    break

                await asyncio.sleep(10)

        # Your spacing choice (60s) stays here
        await asyncio.sleep(60)

async def run_entire_code():
    sem = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
    csv_lock = asyncio.Lock()

    tasks = []
    for prompt_id, variants in PROMPTS_VARIANTS.items():
        for variant_id, variant_text in variants:
            tasks.append(asyncio.create_task(run_variant(prompt_id, variant_id, variant_text, sem, csv_lock)))

    await asyncio.gather(*tasks)
    print(f"\nDone. Results saved to {CSV_PATH}")

await run_entire_code()



prompt 1:1 collecting 30 runs

prompt 1:2 collecting 30 runs

prompt 1:3 collecting 30 runs

prompt 1:4 collecting 30 runs

prompt 1:5 collecting 30 runs

prompt 2:1 collecting 30 runs

prompt 2:2 collecting 30 runs

prompt 2:3 collecting 30 runs

prompt 2:4 collecting 30 runs

prompt 2:5 collecting 30 runs

prompt 3:1 collecting 30 runs

prompt 3:2 collecting 30 runs

prompt 3:3 collecting 30 runs

prompt 3:4 collecting 30 runs

prompt 3:5 collecting 30 runs

prompt 4:1 collecting 30 runs

prompt 4:2 collecting 30 runs

prompt 4:3 collecting 30 runs

prompt 4:4 collecting 30 runs

prompt 4:5 collecting 30 runs

prompt 5:1 collecting 30 runs

prompt 5:2 collecting 30 runs

prompt 5:3 collecting 30 runs

prompt 5:4 collecting 30 runs

prompt 5:5 collecting 30 runs

prompt 6:1 collecting 30 runs

prompt 6:2 collecting 30 runs

prompt 6:3 collecting 30 runs

prompt 6:4 collecting 30 runs

prompt 6:5 collecting 30 runs

prompt 7:1 collecting 30 runs

prompt 7:2 collecting 30 runs

prompt 

/tmp/ipython-input-1768532453.py:111: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.datetime.utcnow().isoformat(),


Saved 1:3 run 01 (tokens 2080)
Saved 1:1 run 01 (tokens 2161)
Saved 1:5 run 01 (tokens 3371)
Saved 1:2 run 01 (tokens 3442)
Saved 1:4 run 01 (tokens 3382)
Saved 2:1 run 01 (tokens 1856)
Saved 2:2 run 01 (tokens 3113)
Saved 2:4 run 01 (tokens 2380)
Saved 3:1 run 01 (tokens 1819)
Saved 2:3 run 01 (tokens 2418)
Saved 2:5 run 01 (tokens 2852)
Saved 3:3 run 01 (tokens 2294)
Saved 3:4 run 01 (tokens 2841)
Saved 4:1 run 01 (tokens 1831)
Error 3:2 run 01 attempt 1: Error code: 400 - {'error': {'message': "Invalid prompt: we've limited access to this content for safety reasons. This type of information may be used to benefit or to harm people. We are continuously refining our work in this area, and you can read more about our approach in our blog post (https://openai.com/index/preparing-for-future-ai-capabilities-in-biology) and Model Spec (https://openai.com/index/introducing-the-model-spec).", 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_prompt'}}
Saved 3:5 run 01 (tokens 